QUANTIZE THE MODELS

The following script was used in a wsl ubuntu env because of problems with the library in windows

In [ ]:
import os
os.environ["YOLO_CONFIG_DIR"] = os.path.expanduser("~/UltralyticsConfig")
os.makedirs(os.environ["YOLO_CONFIG_DIR"], exist_ok=True)
import types
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.nn.modules import Conv, Bottleneck

class C2f_v2(nn.Module):
    """CSP Bottleneck with 2 convolutions - Modified for pruning"""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__()
        self.c = int(c2 * e)
        self.cv0 = Conv(c1, self.c, 1, 1)
        self.cv1 = Conv(c1, self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)
        self.m = nn.ModuleList(
            Bottleneck(self.c, self.c, shortcut, g, k=((3, 3), (3, 3)), e=1.0)
    
            for _ in range(n)
        )

    def forward(self, x):
        y = [self.cv0(x), self.cv1(x)]
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))

PT = "/mnt/c/Users/alijo/OneDrive/Desktop/BachelorThesis/BachelorThesis/Optimization/runs/detect/BN_SVD_0.9/weights/best.pt"
DATA = "/mnt/c/Users/alijo/OneDrive/Desktop/BachelorThesis/BachelorThesis/calib_train_wsl.yaml"

m = YOLO(PT)

def _no_fuse(self, verbose=True):
    return self

m.model.fuse = types.MethodType(_no_fuse, m.model)

out = m.export(format="tflite", imgsz=340, int8=True, data=DATA, fraction=1.0, simplify=True)
print("✅ Exported:", out)

In [3]:
import os
from pathlib import Path

# --------- CONFIG (edit these) ----------
MODEL_TFLITE = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesisCode\BachelorThesis\Optimization\HYB_ONESHOTL2_BN\YOLO_V8_BN_0.7_HYB_L2ONE_BN_PRUNE\step_0_finetune\weights\best_saved_model\best_full_integer_quant.tflite"
IMAGE_PATH  = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesisCode\BachelorThesis\LeafminerOnly\test\images\310195_20210914_2_2_b6_3_2_12_1_37_jpg.rf.b982cbade3d59441dbbd980fffdd12f3.jpg"

OUT_DIR = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesisCode\BachelorThesis\DataSet"  # will create Inc/ Src inside
# --------------------------------------

def write_model_c_array(model_path: str, out_dir: str):
    out_dir = Path(out_dir)
    inc = out_dir / "Inc"
    src = out_dir / "Src"
    inc.mkdir(parents=True, exist_ok=True)
    src.mkdir(parents=True, exist_ok=True)

    data = Path(model_path).read_bytes()

    # Header
    (inc / "model_data.h").write_text(
        "#pragma once\n"
        "#include <cstdint>\n"
        "extern const unsigned char g_model[];\n"
        "extern const unsigned int g_model_len;\n"
    )

    # Source (C++)
    # Use alignas(16) for better DMA/CPU access patterns on some MCUs
    hex_bytes = ", ".join(f"0x{b:02X}" for b in data)
    (src / "model_data.cc").write_text(
        '#include "model_data.h"\n'
        '#include <cstddef>\n\n'
        "alignas(16) const unsigned char g_model[] = {\n"
        f"{hex_bytes}\n"
        "};\n"
        f"const unsigned int g_model_len = {len(data)};\n"
    )

    print(f"[OK] Wrote model_data.h / model_data.cc  (bytes={len(data)})")

def inspect_tflite_io(model_path: str):
    import tensorflow as tf
    itp = tf.lite.Interpreter(model_path=model_path)
    itp.allocate_tensors()
    in0 = itp.get_input_details()[0]
    outs = itp.get_output_details()
    print("INPUT:", in0["shape"], in0["dtype"], "quant:", in0.get("quantization_parameters", {}))
    print("OUTPUTS:")
    for o in outs:
        print(" ", o["index"], o["shape"], o["dtype"], "quant:", o.get("quantization_parameters", {}))
    return in0

def load_and_prepare_image(image_path: str, in_shape, in_dtype, qparams):
    # Uses Pillow. If not installed: pip install pillow
    from PIL import Image
    import numpy as np

    # Expect [1,H,W,C]
    H, W = int(in_shape[1]), int(in_shape[2])
    img = Image.open(image_path).convert("RGB").resize((W, H))
    arr = np.array(img)  # uint8 [H,W,3]

    if in_dtype == np.uint8:
        # common quantized input
        out = arr.astype(np.uint8)
    elif in_dtype == np.int8:
        # map uint8 0..255 into int8 using input quantization params
        # If quant params are present, best is: q = round(x/scale + zero_point)
        # x should be float input in model's expected range.
        scale = qparams["scales"][0] if "scales" in qparams and len(qparams["scales"]) else None
        zp    = qparams["zero_points"][0] if "zero_points" in qparams and len(qparams["zero_points"]) else None

        if scale is None or zp is None:
            # fallback (works for many "uint8 centered to int8" models)
            out = (arr.astype(np.int16) - 128).clip(-128, 127).astype(np.int8)
        else:
            # assume model expects float [0..1] before quantization (common)
            x = arr.astype(np.float32) / 255.0
            q = np.round(x / scale + zp).astype(np.int32)
            out = np.clip(q, -128, 127).astype(np.int8)
    else:
        raise RuntimeError(f"Unsupported input dtype: {in_dtype}")

    # Add batch dimension
    out = out.reshape((1, H, W, 3))
    return out

def write_image_c_array(img_np, out_dir: str):
    import numpy as np
    out_dir = Path(out_dir)
    inc = out_dir / "Inc"
    src = out_dir / "Src"
    inc.mkdir(parents=True, exist_ok=True)
    src.mkdir(parents=True, exist_ok=True)

    flat = img_np.flatten()

    # Header
    (inc / "image_data.h").write_text(
        "#pragma once\n"
        "#include <cstdint>\n"
        "extern const unsigned char g_image[];\n"
        "extern const unsigned int g_image_len;\n"
    )

    # Store as bytes (unsigned char) regardless of uint8/int8; copy raw bytes into input tensor
    raw = flat.tobytes()
    hex_bytes = ", ".join(f"0x{b:02X}" for b in raw)

    (src / "image_data.cc").write_text(
        '#include "image_data.h"\n'
        '#include <cstddef>\n\n'
        "alignas(16) const unsigned char g_image[] = {\n"
        f"{hex_bytes}\n"
        "};\n"
        f"const unsigned int g_image_len = {len(raw)};\n"
    )

    print(f"[OK] Wrote image_data.h / image_data.cc (bytes={len(raw)})")

def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    # 1) model -> C array
    write_model_c_array(MODEL_TFLITE, OUT_DIR)

    # 2) inspect IO and make image array
    in0 = inspect_tflite_io(MODEL_TFLITE)
    import numpy as np
    in_shape = in0["shape"]
    in_dtype = in0["dtype"]
    qparams = in0.get("quantization_parameters", {})

    img_np = load_and_prepare_image(IMAGE_PATH, in_shape, in_dtype, qparams)
    write_image_c_array(img_np, OUT_DIR)

    print("\nDone. Copy OUT/Inc and OUT/Src into your CubeIDE project Example/Inc and Example/Src.")

if __name__ == "__main__":
    main()
    


[OK] Wrote model_data.h / model_data.cc  (bytes=990206)
INPUT: [  1 352 352   3] <class 'numpy.int8'> quant: {'scales': array([0.00392157], dtype=float32), 'zero_points': array([-128]), 'quantized_dimension': 0}
OUTPUTS:
  427 [   1    5 2541] <class 'numpy.int8'> quant: {'scales': array([0.00663836], dtype=float32), 'zero_points': array([-125]), 'quantized_dimension': 0}
[OK] Wrote image_data.h / image_data.cc (bytes=371712)

Done. Copy OUT/Inc and OUT/Src into your CubeIDE project Example/Inc and Example/Src.
